# Hybrid Truck Load Baseline на Kaggle

Обучение и инференс гибридной модели (CNN + 277 признаков) с автоматическим поиском датасета и segmentation-весов.

In [1]:
# 1. Проверка GPU и установка библиотек
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q albumentations timm segmentation-models-pytorch pandas


Устройство: cuda
GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.1 MB/s eta 0:00:00


In [2]:
# 2. Поиск датасета в Kaggle Input и копирование исходного кода
import shutil
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
candidates = []
for split_path in KAGGLE_INPUT.rglob("DataSet/train/train_split.csv"):
    dataset_root = split_path.parents[2]
    if (dataset_root / "src/baseline.py").is_file():
        candidates.append(dataset_root)

if not candidates:
    raise FileNotFoundError(
        "Не найден Kaggle Dataset с DataSet/train/train_split.csv и src/baseline.py"
    )
if len(candidates) > 1:
    print("Найдено несколько датасетов:", [str(path) for path in candidates])

DATASET_ROOT = candidates[0]
SRC_INPUT = DATASET_ROOT / "src"
WORK_SRC = Path("/kaggle/working/src")
if WORK_SRC.exists():
    shutil.rmtree(WORK_SRC)
shutil.copytree(SRC_INPUT, WORK_SRC)

print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"WORK_SRC: {WORK_SRC}")
%cd /kaggle/working/src
!ls -la


DATASET_ROOT: /kaggle/input/datasets/kirillpovasin/truck-dataset
WORK_SRC: /kaggle/working/src
/kaggle/working/src
total 56
drwxr-xr-x 5 root root 4096 Sep 20 08:01 .
drwxr-xr-x 4 root root 4096 Sep 20 08:02 ..
-rw-r--r-- 1 root root 3995 Sep 20 08:01 baseline.py
drwxr-xr-x 3 root root 4096 Sep 20 08:01 CNN
-rw-r--r-- 1 root root 5434 Sep 20 08:01 debug.py
drwxr-xr-x 2 root root 4096 Sep 20 08:01 hybrid
drwxr-xr-x 4 root root 4096 Sep 20 08:01 manual
-rw-r--r-- 1 root root 1578 Sep 20 08:01 predict.py
-rw-r--r-- 1 root root 5073 Sep 20 08:01 report.py
-rw-r--r-- 1 root root  961 Sep 20 08:01 submission.py
-rw-r--r-- 1 root root 6527 Sep 20 08:01 train.py


In [3]:
# 3. Настройка путей к данным, весам и артефактам
from pathlib import Path

# Входные данные
TRAIN_SPLIT = DATASET_ROOT / "DataSet/train/train_split.csv"
VAL_SPLIT = DATASET_ROOT / "DataSet/train/validation_split.csv"
TRAIN_IMAGES = DATASET_ROOT / "DataSet/train/images"
TEST_CSV = DATASET_ROOT / "DataSet/test/test.csv"
TEST_IMAGES = DATASET_ROOT / "DataSet/test/images"

# Веса находятся в скопированном src проекта.
TRUCK_WEIGHTS = WORK_SRC / "manual/truck_segmentation/models/best_unet_resnet18.pth"
FLOOR_WEIGHTS = WORK_SRC / "manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt"

MODELS_DIR = Path("/kaggle/working/models")
OUTPUT_DIR = Path("/kaggle/working/output_csv")
CACHE_DIR = Path("/kaggle/working/feature_cache")
for directory in (MODELS_DIR, OUTPUT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "hybrid_model.npz"
ERRORS_PATH = MODELS_DIR / "val_errors.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

required_paths = {
    "train_split": TRAIN_SPLIT,
    "validation_split": VAL_SPLIT,
    "train_images": TRAIN_IMAGES,
    "test_csv": TEST_CSV,
    "test_images": TEST_IMAGES,
    "truck_weights": TRUCK_WEIGHTS,
    "floor_weights": FLOOR_WEIGHTS,
}
for name, path in required_paths.items():
    print(f"{name}: {path} -> {'OK' if path.exists() else 'НЕ НАЙДЕН'}")
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Не найдены обязательные пути: {missing}")


train_split: /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/train_split.csv -> OK
validation_split: /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/validation_split.csv -> OK
train_images: /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/images -> OK
test_csv: /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/test.csv -> OK
test_images: /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/images -> OK
truck_weights: /kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth -> OK
floor_weights: /kaggle/working/src/manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt -> OK


In [4]:
# 4. Обучение модели
import subprocess
import sys

BASELINE = WORK_SRC / "baseline.py"
train_command = [
    sys.executable,
    str(BASELINE),
    "train",
    "--method", "hybrid",
    "--train-split", str(TRAIN_SPLIT),
    "--validation-split", str(VAL_SPLIT),
    "--images", str(TRAIN_IMAGES),
    "--model", str(MODEL_PATH),
    "--errors-output", str(ERRORS_PATH),
    "--feature-cache", str(CACHE_DIR),
    "--truck-weights", str(TRUCK_WEIGHTS),
    "--floor-weights", str(FLOOR_WEIGHTS),
    "--device", DEVICE,
    "--batch-size", "16",
    "--num-workers", "2",
    "--phase1-epochs", "6",
    "--phase2-epochs", "18",
    "--patience", "6",
]
print("Запуск:", " ".join(train_command))
subprocess.run(train_command, check=True)


Запуск: /usr/bin/python3 /kaggle/working/src/baseline.py train --method hybrid --train-split /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/train_split.csv --validation-split /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/validation_split.csv --images /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/images --model /kaggle/working/models/hybrid_model.npz --errors-output /kaggle/working/models/val_errors.csv --feature-cache /kaggle/working/feature_cache --truck-weights /kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth --floor-weights /kaggle/working/src/manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt --device cuda --batch-size 16 --num-workers 2 --phase1-epochs 6 --phase2-epochs 18 --patience 6
[SEGMENTATION] loading model from /kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth
[SEGMENTATION] ready | device=cuda | size=256
ROI и признаки гибрида: 50/573
ROI и при

{"epoch": 1, "phase": 1, "train_loss": 17.287905188755214, "val_mae": 14.14391956629453}
{"epoch": 2, "phase": 1, "train_loss": 11.709909677089405, "val_mae": 12.227825369868246}
{"epoch": 3, "phase": 1, "train_loss": 10.827035935553372, "val_mae": 11.054239886623996}
{"epoch": 4, "phase": 1, "train_loss": 9.898063922635755, "val_mae": 10.92607129370416}
{"epoch": 5, "phase": 1, "train_loss": 9.20496122749688, "val_mae": 11.055382588526586}
{"epoch": 6, "phase": 1, "train_loss": 8.778334006797165, "val_mae": 10.927457536017144}
{"epoch": 7, "phase": 2, "train_loss": 10.42721209600958, "val_mae": 10.894542183909383}
{"epoch": 8, "phase": 2, "train_loss": 9.042357263348698, "val_mae": 10.473951679843289}
{"epoch": 9, "phase": 2, "train_loss": 7.934361404565409, "val_mae": 10.61664628982544}
{"epoch": 10, "phase": 2, "train_loss": 7.933821032392625, "val_mae": 10.446003293657636}
{"epoch": 11, "phase": 2, "train_loss": 6.892332534723465, "val_mae": 10.666900311316644}
{"epoch": 12, "phase

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/src/baseline.py', 'train', '--method', 'hybrid', '--train-split', '/kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/train_split.csv', '--validation-split', '/kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/validation_split.csv', '--images', '/kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/train/images', '--model', '/kaggle/working/models/hybrid_model.npz', '--errors-output', '/kaggle/working/models/val_errors.csv', '--feature-cache', '/kaggle/working/feature_cache', '--truck-weights', '/kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth', '--floor-weights', '/kaggle/working/src/manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt', '--device', 'cuda', '--batch-size', '16', '--num-workers', '2', '--phase1-epochs', '6', '--phase2-epochs', '18', '--patience', '6'], returncode=0)

In [5]:
# 5. Метрики валидации и отчет
import json
import pandas as pd

report_path = MODELS_DIR / "hybrid_model.json"
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)
    print("=== Результаты валидации ===")
    print(json.dumps(report.get("hybrid", {}), indent=2, ensure_ascii=False))
    print(f"Лучшая эпоха: {report.get('configuration', {}).get('best_epoch')}")

if ERRORS_PATH.exists():
    df_err = pd.read_csv(ERRORS_PATH)
    print("\nТоп-5 объектов с наибольшей ошибкой:")
    display(df_err.head(5))

=== Результаты валидации ===
{
  "mae": 9.409679012698726,
  "within_10_pct": 65.03496503496503,
  "count": 143
}
Лучшая эпоха: 13

Топ-5 объектов с наибольшей ошибкой:


,image_id,actual_load_pct,predicted_load_pct,absolute_error_pct
0,img_5b1f259be98ea8ea4e62,70.0,22.497894,47.502106
1,img_386761fc0ae1c50d4df8,40.0,81.141014,41.141014
2,img_c3dedb39849287c798bc,50.0,11.253481,38.746519
3,img_a9d11f8bc4a69cb3f21d,70.0,32.386726,37.613274
4,img_75a939a9c8d17afbe97f,90.0,55.441402,34.558598


In [6]:
# 6. Инференс на тестовой выборке
predict_command = [
    sys.executable,
    str(BASELINE),
    "predict",
    "--model", str(MODEL_PATH),
    "--test-csv", str(TEST_CSV),
    "--images", str(TEST_IMAGES),
    "--output", str(SUBMISSION_PATH),
    "--feature-cache", str(CACHE_DIR),
    "--truck-weights", str(TRUCK_WEIGHTS),
    "--floor-weights", str(FLOOR_WEIGHTS),
    "--device", DEVICE,
    "--batch-size", "16",
    "--num-workers", "2",
]
print("Запуск:", " ".join(predict_command))
subprocess.run(predict_command, check=True)


Запуск: /usr/bin/python3 /kaggle/working/src/baseline.py predict --model /kaggle/working/models/hybrid_model.npz --test-csv /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/test.csv --images /kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/images --output /kaggle/working/output_csv/submission.csv --feature-cache /kaggle/working/feature_cache --truck-weights /kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth --floor-weights /kaggle/working/src/manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt --device cuda --batch-size 16 --num-workers 2
[SEGMENTATION] loading model from /kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth
[SEGMENTATION] ready | device=cuda | size=256
ROI и признаки гибрида: 50/307
ROI и признаки гибрида: 100/307
ROI и признаки гибрида: 150/307
ROI и признаки гибрида: 200/307
ROI и признаки гибрида: 250/307
ROI и признаки гибрида: 300/307
ROI и признаки гибрида: 307/307
S

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/src/baseline.py', 'predict', '--model', '/kaggle/working/models/hybrid_model.npz', '--test-csv', '/kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/test.csv', '--images', '/kaggle/input/datasets/kirillpovasin/truck-dataset/DataSet/test/images', '--output', '/kaggle/working/output_csv/submission.csv', '--feature-cache', '/kaggle/working/feature_cache', '--truck-weights', '/kaggle/working/src/manual/truck_segmentation/models/best_unet_resnet18.pth', '--floor-weights', '/kaggle/working/src/manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt', '--device', 'cuda', '--batch-size', '16', '--num-workers', '2'], returncode=0)

In [7]:
# 7. Проверка сформированного файла submission.csv
import pandas as pd

if SUBMISSION_PATH.exists():
    sub_df = pd.read_csv(SUBMISSION_PATH)
    print(f"Размер предсказаний: {sub_df.shape}")
    print(f"Количество пропусков (NaN): {sub_df.isna().sum().sum()}")
    display(sub_df.head(10))
else:
    print("Файл submission.csv не найден!")

Размер предсказаний: (307, 2)
Количество пропусков (NaN): 0


,image_id,load_pct
0,img_00b135ede40393ac4459,98.430473
1,img_024af7d284b5259e46db,96.950409
2,img_031cb4706aa99c2c29d7,97.611488
3,img_037814d0bd90cdd3f519,40.297741
4,img_03bed56db6bd8389fdce,94.396286
5,img_04385169354be79e7122,8.424207
6,img_04c2f14a79df14d0a0d8,14.825842
7,img_05a3bf0b269db86a5ad6,14.842237
8,img_061bc6606624bbace682,11.841879
9,img_07f208b92285b85dbfe4,12.845376
